In [3]:
!pip install kaggle
from google.colab import files

# Upload Kaggle API key (kaggle.json)
files.upload()

# Move kaggle.json to the correct directory
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [4]:
!kaggle datasets download -d sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset
!unzip deep-fake-detection-dfd-entire-original-dataset.zip -d /content/dataset/videos

Dataset URL: https://www.kaggle.com/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset
License(s): MIT
100% 22.5G/22.5G [17:22<00:00, 24.3MB/s]
100% 22.5G/22.5G [17:22<00:00, 23.2MB/s]
Archive:  deep-fake-detection-dfd-entire-original-dataset.zip
  inflating: /content/dataset/videos/DFD_manipulated_sequences/DFD_manipulated_sequences/01_02__exit_phone_room__YVGY8LOK.mp4  
  inflating: /content/dataset/videos/DFD_manipulated_sequences/DFD_manipulated_sequences/01_02__hugging_happy__YVGY8LOK.mp4  
  inflating: /content/dataset/videos/DFD_manipulated_sequences/DFD_manipulated_sequences/01_02__meeting_serious__YVGY8LOK.mp4  
  inflating: /content/dataset/videos/DFD_manipulated_sequences/DFD_manipulated_sequences/01_02__outside_talking_still_laughing__YVGY8LOK.mp4  
  inflating: /content/dataset/videos/DFD_manipulated_sequences/DFD_manipulated_sequences/01_02__secret_conversation__YVGY8LOK.mp4  
  inflating: /content/dataset/videos/DFD_manipulated_sequences/DFD_manipul

In [5]:
!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip deepfake-and-real-images.zip -d /content/dataset/images

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5499.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_55.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_550.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5500.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5501.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5502.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5503.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5504.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5505.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5506.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5507.jpg  
  inflating: /content/dataset/images/Dataset/Validation/Real/real_5508.jpg  
  inflating: /content/datase

In [7]:
import os

print("Video Dataset Files:", os.listdir("/content/dataset/videos")[:10])
print("Image Dataset Files:", os.listdir("/content/dataset/images")[:10])

Video Dataset Files: ['DFD_manipulated_sequences', 'sample_videos.zip', 'DFD_original sequences']
Image Dataset Files: ['Dataset', '.ipynb_checkpoints']


In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "/content/dataset/images/Dataset/Train"
test_dir = "/content/dataset/images/Dataset/Test"

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    subset="validation"
)

test_generator = datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"
)

Found 112002 images belonging to 2 classes.
Found 28000 images belonging to 2 classes.
Found 10905 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model = Sequential([
    Input(shape=(224, 224, 3)),  
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)                    │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 111, 111, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 109, 109, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 54, 54, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 52, 52, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 26, 26, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 86528)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │      11,075,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10 
)

Epoch 1/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 317s 89ms/step - accuracy: 0.7912 - loss: 0.4289 - val_accuracy: 0.9070 - val_loss: 0.2290
Epoch 2/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 227s 65ms/step - accuracy: 0.9253 - loss: 0.1836 - val_accuracy: 0.9349 - val_loss: 0.1576
Epoch 3/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 222s 63ms/step - accuracy: 0.9479 - loss: 0.1291 - val_accuracy: 0.9394 - val_loss: 0.1484
Epoch 4/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 290s 72ms/step - accuracy: 0.9599 - loss: 0.1002 - val_accuracy: 0.9487 - val_loss: 0.1354
Epoch 5/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 246s 67ms/step - accuracy: 0.9675 - loss: 0.0813 - val_accuracy: 0.9486 - val_loss: 0.1394
Epoch 6/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 268s 77ms/step - accuracy: 0.9729 - loss: 0.0686 - val_accuracy: 0.9497 - val_loss: 0.1639
Epoch 7/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 224s 64ms/step - accuracy: 0.9773 - loss: 0.0579 - val_accuracy: 0.9518 - val_loss: 0.1579
Epoch 8/10
3501/3501 ━━━━━━━━━━━━━━━━━━━━ 229s 66ms/step - accuracy: 

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.4f}")

341/341 ━━━━━━━━━━━━━━━━━━━━ 18s 52ms/step - accuracy: 0.8703 - loss: 0.5964
Test Accuracy: 0.8706


In [12]:
model.save("/content/deepfake_model.h5")

In [ ]:
import IPython.display as display

output_path='/content/dataset/videos/DFD_original sequences'
video_files = os.listdir(output_path)
print("Available video files:", video_files[:100])  # عرض أول 10 فيديوهات فقط
video_path = os.path.join(output_path, video_files[0])
display.Video(video_path)


Available video files: ['10__podium_speech_happy.mp4', '08__exit_phone_room.mp4', '10__talking_against_wall.mp4', '27__meeting_serious.mp4', '23__talking_against_wall.mp4', '18__exit_phone_room.mp4', '23__exit_phone_room.mp4', '01__walking_down_street_outside_angry.mp4', '03__meeting_serious.mp4', '23__outside_talking_still_laughing.mp4', '24__secret_conversation.mp4', '15__talking_angry_couch.mp4', '15__walking_down_street_outside_angry.mp4', '09__walking_down_street_outside_angry.mp4', '19__podium_speech_happy.mp4', '16__kitchen_still.mp4', '13__walking_and_outside_surprised.mp4', '26__talking_against_wall.mp4', '20__secret_conversation.mp4', '13__secret_conversation.mp4', '02__walking_and_outside_surprised.mp4', '01__outside_talking_still_laughing.mp4', '22__walking_down_indoor_hall_disgust.mp4', '06__kitchen_still.mp4', '20__kitchen_still.mp4', '20__walking_and_outside_surprised.mp4', '15__walking_outside_cafe_disgusted.mp4', '07__hugging_happy.mp4', '27__talking_against_wall.mp4',

In [ ]:
import cv2

def extract_frames(video_path, output_folder, num_frames=100):
    cap = cv2.VideoCapture(video_path)
    count = 0

    while cap.isOpened() and count < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame_path = os.path.join(output_folder, f"frame_{count}.jpg")
        cv2.imwrite(frame_path, frame)
        count += 1

    cap.release()
    print(f"Extracted {count} frames from {video_path}")

frames_output_path = "/content/extracted_frames"
os.makedirs(frames_output_path, exist_ok=True)

extract_frames(video_path, frames_output_path)

Extracted 100 frames from /content/dataset/videos/DFD_original sequences/10__podium_speech_happy.mp4


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

model = tf.keras.models.load_model("/content/deepfake_model.h5")

test_frame = os.path.join(frames_output_path, "frame_0.jpg")
img = image.load_img(test_frame, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)[0][0]
print("Prediction:", "Fake" if prediction > 0.5 else "Real")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
Prediction: Fake
